In [1]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa 
import pandas as pd
import numpy as np 
from models import load_scene_model, load_face_model, load_audio_model, load_text_glove_model

c:\Users\SIA\anaconda3\envs\deepface-env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Load models

In [2]:
scene_model = load_scene_model()
face_model  = load_face_model()
audio_model = load_audio_model()
text_model  = load_text_glove_model()

### Freeze layers

In [3]:
for layer in scene_model.layers:
    layer.trainable = False 
    layer._name = 'Scene_' + layer._name
scene_model.summary()

Model: "scene_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Scene_Input (InputLayer)       [(None, 10, 224, 22  0           []                               
                                4, 3)]                                                            
                                                                                                  
 Scene_Rescaling (TimeDistribut  (None, 10, 224, 224  0          ['Scene_Input[0][0]']            
 ed)                            , 3)                                                              
                                                                                                  
 Scene_time_distributed (TimeDi  (None, 10, 224, 224  0          ['Scene_Input[0][0]']            
 stributed)                     , 3)                                                    

In [4]:
for layer in face_model.layers:
    layer.trainable = False 
    layer._name = 'Face_' + layer._name
face_model.summary()

Model: "face_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Face_Input (InputLayer)        [(None, 10, 224, 22  0           []                               
                                4, 3)]                                                            
                                                                                                  
 Face_Rescaling (TimeDistribute  (None, 10, 224, 224  0          ['Face_Input[0][0]']             
 d)                             , 3)                                                              
                                                                                                  
 Face_time_distributed_2 (TimeD  (None, 10, 224, 224  0          ['Face_Input[0][0]']             
 istributed)                    , 3)                                                     

In [5]:
for layer in audio_model.layers:
    layer.trainable = False
    layer._name = 'Audio_' + layer._name
audio_model.summary()

Model: "audio_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Audio_input_2 (InputLayer)  [(None, 15, 128)]         0         
                                                                 
 Audio_conv1d (Conv1D)       (None, 14, 32)            8224      
                                                                 
 Audio_dropout_6 (Dropout)   (None, 14, 32)            0         
                                                                 
 Audio_conv1d_1 (Conv1D)     (None, 13, 64)            4160      
                                                                 
 Audio_dropout_7 (Dropout)   (None, 13, 64)            0         
                                                                 
 Audio_lstm_8 (LSTM)         (None, 13, 512)           1181696   
                                                                 
 Audio_lstm_9 (LSTM)         (None, 256)               

In [6]:
for layer in text_model.layers:
    layer.trainable = False 
    layer._name = 'Text_' + layer._name
text_model.summary()

Model: "text_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Text_input_3 (InputLayer)      [(None, 50)]         0           []                               
                                                                                                  
 Text_embedding (Embedding)     (None, 50, 100)      8900        ['Text_input_3[0][0]']           
                                                                                                  
 Text_conv1d_2 (Conv1D)         (None, 48, 16)       4816        ['Text_embedding[0][0]']         
                                                                                                  
 Text_conv1d_4 (Conv1D)         (None, 48, 32)       9632        ['Text_embedding[0][0]']         
                                                                                         

### Build model

In [7]:
scene_inputs = keras.layers.Input(shape=(10,224,224,3), name='Scene_input')
face_inputs  = keras.layers.Input(shape=(10,224,224,3), name='Face_input')
audio_inputs = keras.layers.Input(shape=(15,128), name='Audio_input')
text_inputs  = keras.layers.Input(shape=(50), name='Text_input')

x = scene_model.layers[-13].output
y = face_model.layers[-13].output

v = keras.layers.Average()([x,y])
v = keras.layers.Dense(64, activation='relu')(v)

a = audio_model.layers[-6].output
t = text_model.layers[-8].output
t = keras.layers.Dense(64, activation='relu')(t)

a1 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(v,a) # video->audio
a2 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(a,v) # audio->video

a3 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(v,t) # video->text
a4 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(t,v) # text->video

a5 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(a,t) # audio->text
a6 = keras.layers.MultiHeadAttention(num_heads=2, key_dim=64)(t, a) # text-audio



o = keras.layers.Concatenate(axis=1)([a1, a2, a3, a4, a5, a6])
o = keras.layers.GlobalAveragePooling1D()(o)


o = keras.layers.Dense(64, activation='relu')(o)
o = keras.layers.Dense(5, activation='sigmoid')(o)

atten_model = keras.models.Model(inputs=[scene_model.input, face_model.input, audio_model.input, text_model.input], outputs=o)

atten_model.compile(loss='mse', optimizer=tfa.optimizers.RectifiedAdam(), metrics=['mae'])




In [8]:
import datetime
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=0)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/cross_atten/attention.t5',
                             monitor='val_mae',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=0)

### Load data

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

# Train
scene_train_ds = tf.data.experimental.load('./data/fullscene/train_ds/')
face_train_ds  = tf.data.experimental.load('./data/faces/train_ds/')
audio_train_ds = tf.data.experimental.load('./data/audio/train_ds/')
text_train_ds  = tf.data.experimental.load('./data/text/train_ds/').batch(batch_size=32)

scene_xtrain = scene_train_ds.map(lambda x,y: x)
face_xtrain  = face_train_ds.map(lambda x,y: x)
audio_xtrain = audio_train_ds.map(lambda x,y: x)
text__xtrain = text_train_ds.map(lambda x,y: x)
y_train      = scene_train_ds.map(lambda x,y: y)

train_ds = tf.data.Dataset.zip(((scene_xtrain, face_xtrain, audio_xtrain, text__xtrain), y_train)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)


# Valid
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)

train_ds, valid_ds

Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


Instructions for updating:
Use `tf.data.Dataset.load(...)` instead.


(<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Train

In [10]:
history = atten_model.fit(train_ds, validation_data=valid_ds, batch_size=32, epochs=100, callbacks=[early_stopping, check_point], verbose=1)

Epoch 1/100
1/1 [==============================] - 67s 67s/step - loss: 0.0306 - mae: 0.1629 - val_loss: 0.0121 - val_mae: 0.1057
Epoch 2/100
1/1 [==============================] - 32s 32s/step - loss: 0.0307 - mae: 0.1629 - val_loss: 0.0121 - val_mae: 0.1056
Epoch 3/100
1/1 [==============================] - 19s 19s/step - loss: 0.0307 - mae: 0.1629 - val_loss: 0.0121 - val_mae: 0.1056
Epoch 4/100
1/1 [==============================] - 35s 35s/step - loss: 0.0308 - mae: 0.1629 - val_loss: 0.0121 - val_mae: 0.1056
Epoch 5/100
1/1 [==============================] - 30s 30s/step - loss: 0.0309 - mae: 0.1639 - val_loss: 0.0121 - val_mae: 0.1056
Epoch 6/100
1/1 [==============================] - 31s 31s/step - loss: 0.0305 - mae: 0.1625 - val_loss: 0.0120 - val_mae: 0.1052
Epoch 7/100
1/1 [==============================] - 14s 14s/step - loss: 0.0305 - mae: 0.1625 - val_loss: 0.0119 - val_mae: 0.1048
Epoch 8/100
1/1 [==============================] - 16s 16s/step - loss: 0.0306 - mae: 0.16

### Load weights

In [11]:
atten_model.load_weights('./weights/cross_atten/attention.t5')

## Evaluation

### Validation data

In [12]:
AUTOTUNE = tf.data.AUTOTUNE
scene_valid_ds = tf.data.experimental.load('./data/fullscene/val_ds/')
face_valid_ds  = tf.data.experimental.load('./data/faces/val_ds/')
audio_valid_ds = tf.data.experimental.load('./data/audio/val_ds') 
text_valid_ds  = tf.data.experimental.load('./data/text/val_ds/').batch(batch_size=32)

scene_xvalid = scene_valid_ds.map(lambda x,y: x)
face_xvalid  = face_valid_ds.map(lambda x,y: x)
audio_xvalid = audio_valid_ds.map(lambda x,y: x)
text_xvalid  = text_valid_ds.map(lambda x,y: x)
y_valid      = scene_valid_ds.map(lambda x,y: y)

valid_ds = tf.data.Dataset.zip(((scene_xvalid, face_xvalid, audio_xvalid, text_xvalid), y_valid)).prefetch(buffer_size=AUTOTUNE)


In [13]:
from sklearn.metrics import mean_absolute_error 

y_true = np.concatenate([y for x,y in valid_ds], axis=0)
y_pred = atten_model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 9s 9s/step


(array([96.43932 , 99.789055, 97.62242 , 91.27495 , 92.77889 ],
       dtype=float32),
 95.58092728257179)

### Test data

In [14]:
scene_test_ds = tf.data.experimental.load('./data/fullscene/test_ds/')
face_test_ds  = tf.data.experimental.load('./data/faces/test_ds/')
audio_test_ds = tf.data.experimental.load('./data/audio/test_ds') 
text_test_ds  = tf.data.experimental.load('./data/text/test_ds/').batch(batch_size=32)


scene_xtest = scene_test_ds.map(lambda x,y: x)
face_xtest  = face_test_ds.map(lambda x,y: x)
audio_xtest = audio_test_ds.map(lambda x,y: x)
text_xtest  = text_test_ds.map(lambda x,y: x)

y_test      = scene_test_ds.map(lambda x,y: y)

test_ds = tf.data.Dataset.zip(((scene_xtest, face_xtest, audio_xtest, text_xtest), y_test)).prefetch(buffer_size=AUTOTUNE)

test_ds

<_PrefetchDataset element_spec=((TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 15, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 50), dtype=tf.int32, name=None)), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>

In [15]:
y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = atten_model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

1/1 [==============================] - 7s 7s/step


(array([90.76331 , 88.08826 , 99.228806, 94.70887 , 95.59043 ],
       dtype=float32),
 93.6759352684021)

In [16]:
import pickle
with open('./histories/attention_cross.pkl', 'wb') as f:
    pickle.dump(history.history, f)